# S2.9 — Catalyst Optimizer
**Date completed:** September 2026  
**Status:** In Progress  
**Interview covered:** Q239 — Catalyst Optimizer phases

In [0]:
# ============================================================
# Cell 2 — Proving Catalyst Optimizer with EXPLAIN
# Goal: See what Catalyst does to your query before executing
# Tool: df.explain() shows the physical execution plan
# ============================================================

import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

# Create sample DataFrame — same structure as RetailPulse orders
schema = StructType([
    StructField("order_id", LongType(), False),
    StructField("customer_id", LongType(), False),
    StructField("city", StringType(), True),
    StructField("order_value", DoubleType(), True),
    StructField("product_category", StringType(), True)
])

data = [
    (1001, 201, "Delhi", 2500.0, "Electronics"),
    (1002, 202, "Mumbai", 1800.0, "Clothing"),
    (1003, 203, "Pune", 3200.0, "Electronics"),
    (1004, 204, "Delhi", 950.0, "Food"),
    (1005, 205, "Chennai", 4100.0, "Electronics"),
    (1006, 206, "Delhi", 750.0, "Clothing"),
    (1007, 207, "Mumbai", 2200.0, "Food")
]

df = spark.createDataFrame(data, schema)

print("=== UNOPTIMISED query (what we write) ===")
unoptimised = df.select("order_id", "city", "order_value",
                         "customer_id", "product_category") \
                .filter(F.col("city") == "Delhi") \
                .groupBy("city") \
                .agg(F.sum("order_value").alias("total_revenue"))

print("Query plan — see what Catalyst does:")
unoptimised.explain()
print()
print("=== RESULT ===")
unoptimised.show()

In [0]:
# ============================================================
# Cell 3 — EXPLAIN EXTENDED: See all 4 Catalyst phases
# Goal: See Parsed, Analysed, Optimised and Physical plans
# ============================================================

print("=== EXPLAIN EXTENDED — All 4 Catalyst phases ===")
print()
unoptimised.explain(extended=True)

## Key Takeaways — S2.9 Catalyst Optimizer

## 4 Phases
| Phase | Name | What happens |
|-------|------|-------------|
| 1 | Parse | Read your code → build AST → unresolved columns |
| 2 | Analyse | Verify columns exist → resolve types → validate |
| 3 | Optimise | Rewrite plan → Column Pruning + Predicate Pushdown |
| 4 | Physical | Choose execution strategy → Photon → execute |

## Two Key Optimisations Proved
**Column Pruning:**
- You selected 5 columns
- Catalyst loaded only 2 (city, order_value)
- Unused columns never touch memory

**Predicate Pushdown:**
- You filtered after select
- Catalyst pushed filter to data loading stage
- Fewer rows processed from the start

## EXPLAIN commands
- df.explain()              → Physical plan only
- df.explain(extended=True) → All 4 phases

## Photon Engine
- Databricks vectorized execution engine
- C++ speed — replaces standard JVM execution
- "Fully supported by Photon" = maximum performance
- Deep coverage in S13

## Key Rule
Write clean readable code.
Catalyst optimises automatically.
Use EXPLAIN to verify what Catalyst did.

#Catalyst is smart — but not magic.
It cannot fix a fundamentally bad design.

Example:
If you join two 10M row tables without any filter
Catalyst cannot invent data that does not exist
It can only optimise what IS in your plan

Your job as senior engineer:
→ Write correct logic
→ Catalyst handles micro-optimisation
→ You handle macro-design decisions